# AI CV Personality Analyzer
## 03 - Feature Engineering

### Project Overview

This notebook converts the cleaned CV/resume text into numerical
features suitable for Machine Learning.

Machine Learning algorithms cannot directly understand raw text.
Therefore, Natural Language Processing (NLP) techniques are used
to represent the text numerically.

### Feature Engineering Strategy

This project uses three types of features:

1. **Word-level TF-IDF**
   - Captures important words and word combinations.
   - Uses unigrams and bigrams.

2. **Character-level TF-IDF**
   - Captures character patterns, word fragments, and linguistic
     characteristics.
   - Uses character n-grams.

3. **Text Statistical Features**
   - Character count
   - Word count
   - Sentence count
   - Average word length
   - Uppercase ratio
   - Digit ratio
   - Punctuation ratio

These features will be combined into a single sparse feature matrix.

### Data Leakage Prevention

The TF-IDF vectorizers will be fitted **only on the training dataset**.

Validation and test datasets will only be transformed using the
vectorizers learned from the training data.

### Output

The final feature matrices will be saved for the model training stage:

- `X_train.npz`
- `X_validation.npz`
- `X_test.npz`

The trained TF-IDF vectorizers will also be saved for later use
when dynamically analyzing new CVs.

## 1. Import Required Libraries

We import the libraries required for:

- Data loading
- TF-IDF feature extraction
- Sparse matrix operations
- Text statistics
- Feature scaling
- Saving feature matrices and vectorizers

In [2]:
# Import pandas for DataFrame operations
import pandas as pd

# Import NumPy for numerical calculations
import numpy as np

# Import regular expressions for text analysis
import re

# Import os for directory and file management
import os

# Import joblib for saving trained vectorizers and feature objects
import joblib

# Import TF-IDF vectorizers for NLP feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# Import sparse matrix utilities
from scipy.sparse import hstack, csr_matrix, save_npz, load_npz

# Import MaxAbsScaler for scaling numerical features
# while preserving sparse matrix compatibility
from sklearn.preprocessing import MaxAbsScaler

# Display all DataFrame columns
pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Define Input and Output Paths

The processed datasets were created in Notebook 02.

We will load:

- `train_processed.parquet`
- `validation_processed.parquet`
- `test_processed.parquet`

The generated feature matrices and vectorizers will be stored inside
the `models/` directory.

The raw dataset will not be modified.

In [3]:
# Define the directory containing processed datasets
PROCESSED_DIR = "../data/processed"

# Define the directory where feature engineering artifacts will be saved
MODELS_DIR = "../models"

# Create the models directory if it does not already exist
os.makedirs(MODELS_DIR, exist_ok=True)

# Define processed dataset paths
TRAIN_PATH = os.path.join(
    PROCESSED_DIR,
    "train_processed.parquet"
)

VALIDATION_PATH = os.path.join(
    PROCESSED_DIR,
    "validation_processed.parquet"
)

TEST_PATH = os.path.join(
    PROCESSED_DIR,
    "test_processed.parquet"
)

print("Paths configured successfully.")
print("Processed data directory:", PROCESSED_DIR)
print("Models directory:", MODELS_DIR)

Paths configured successfully.
Processed data directory: ../data/processed
Models directory: ../models


## 3. Load Processed Datasets

Load the cleaned datasets generated by Notebook 02.

The train, validation, and test splits remain separate.

In [4]:
# Load the processed training dataset
train_df = pd.read_parquet(TRAIN_PATH)

# Load the processed validation dataset
validation_df = pd.read_parquet(VALIDATION_PATH)

# Load the processed test dataset
test_df = pd.read_parquet(TEST_PATH)

# Display dataset sizes
print("Processed datasets loaded successfully.")
print("-" * 50)

print(f"Training Set   : {train_df.shape}")
print(f"Validation Set : {validation_df.shape}")
print(f"Test Set       : {test_df.shape}")

Processed datasets loaded successfully.
--------------------------------------------------
Training Set   : (1578, 6)
Validation Set : (395, 6)
Test Set       : (494, 6)


## 4. Verify Dataset Structure

The processed datasets should contain:

- `O`
- `C`
- `E`
- `A`
- `N`
- `text`

The Big Five columns are the target variables.

The `text` column is the main input for NLP feature engineering.

In [5]:
# Define the Big Five personality target columns
TARGET_COLUMNS = ["O", "C", "E", "A", "N"]

# Verify that all required target columns exist
for column in TARGET_COLUMNS:

    if column not in train_df.columns:
        raise ValueError(
            f"Target column '{column}' is missing from the training dataset."
        )

# Verify that the text column exists
if "text" not in train_df.columns:
    raise ValueError(
        "The 'text' column is missing from the training dataset."
    )

print("Dataset structure verified.")
print("\nFeatures available:")
print(train_df.columns.tolist())

Dataset structure verified.

Features available:
['O', 'C', 'E', 'A', 'N', 'text']


## 5. Separate Text Features and Personality Targets

The `text` column will be transformed into numerical features.

The Big Five personality columns will remain separate as prediction
targets.

We do not use the personality labels to build the TF-IDF vocabulary.

This separation helps prevent target leakage.

In [6]:
# Extract text from each dataset
X_text_train = train_df["text"].fillna("").astype(str)
X_text_validation = validation_df["text"].fillna("").astype(str)
X_text_test = test_df["text"].fillna("").astype(str)

# Extract the Big Five target variables
y_train = train_df[TARGET_COLUMNS].copy()
y_validation = validation_df[TARGET_COLUMNS].copy()
y_test = test_df[TARGET_COLUMNS].copy()

# Display the resulting shapes
print("Text data shapes:")
print("Train      :", X_text_train.shape)
print("Validation :", X_text_validation.shape)
print("Test       :", X_text_test.shape)

print("\nTarget shapes:")
print("Train      :", y_train.shape)
print("Validation :", y_validation.shape)
print("Test       :", y_test.shape)

Text data shapes:
Train      : (1578,)
Validation : (395,)
Test       : (494,)

Target shapes:
Train      : (1578, 5)
Validation : (395, 5)
Test       : (494, 5)


## 6. Inspect Text Before Feature Extraction

Before generating NLP features, inspect a few cleaned text examples.

This confirms that the preprocessing stage produced usable text.

In [7]:
# Display several cleaned CV/text examples
for index, text in enumerate(X_text_train.head(3), start=1):

    print("=" * 80)
    print(f"Training Text Example {index}")
    print("=" * 80)
    print(text[:1000])
    print()

Training Text Example 1
it is wednesday. i can't wait until friday because i am going home to see brandon. i miss him so much. i can't wait to see him. two more days. this has been a very long two weeks. time passes very slowly here. i have a lot of free time on my hands when i am not in class. class. psychology class. psychology is fun so far. it really interests me, and prof. pennebaker is funny. chapter two sort of scared me though. how am i going to remember all of those terms. i didn't even finish reading it because i didn't understand it. but i should have becasue matt said that it was interesting. he was telling me about how they cut some part of a cat's brain out in an experiment. that is weird. the poor cat. matt is weird too. i always wonder if he likes me. he can be so mean when other people are around but so nice when it is just the two of us. i did feel pretty uncomfortable around him today in class. it was weird to sit right next to him. those seats are so close. i wish c

## 7. Word-Level TF-IDF

### What is TF-IDF?

TF-IDF stands for:

**Term Frequency - Inverse Document Frequency**

It assigns higher importance to words that are useful for distinguishing
documents while reducing the importance of words that appear frequently
across many documents.

### Configuration

We will use:

- `ngram_range=(1, 2)` → unigrams and bigrams
- `min_df=2` → ignore extremely rare terms
- `max_df=0.95` → ignore terms appearing in almost every document
- `sublinear_tf=True` → use logarithmic term-frequency scaling

### Leakage Prevention

The vectorizer is fitted only on the training text.

Validation and test data are transformed using the same fitted vocabulary.

In [8]:
# Create the word-level TF-IDF vectorizer
word_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=20000
)

# IMPORTANT:
# Fit the vocabulary ONLY on training data
X_train_word = word_vectorizer.fit_transform(X_text_train)

# Transform validation data using the training vocabulary
X_validation_word = word_vectorizer.transform(
    X_text_validation
)

# Transform test data using the training vocabulary
X_test_word = word_vectorizer.transform(
    X_text_test
)

# Display the resulting dimensions
print("Word-level TF-IDF")
print("-" * 50)

print("Train      :", X_train_word.shape)
print("Validation :", X_validation_word.shape)
print("Test       :", X_test_word.shape)

Word-level TF-IDF
--------------------------------------------------
Train      : (1578, 20000)
Validation : (395, 20000)
Test       : (494, 20000)


## 8. Inspect Word-Level TF-IDF Features

Let's inspect the vocabulary learned from the training dataset.

The vocabulary contains the words and word combinations used
to represent the CV text numerically.

In [9]:
# Get the learned TF-IDF feature names
word_features = word_vectorizer.get_feature_names_out()

# Display the total number of word-level features
print("Total Word TF-IDF Features:", len(word_features))

# Display the first 50 learned features
print("\nSample Features:")
print(word_features[:50])

Total Word TF-IDF Features: 20000

Sample Features:
['00' '00 in' '000' '00am' '10' '10 30' '10 and' '10 minutes' '10 more'
 '10 seconds' '10 years' '100' '1000' '11' '11th' '12' '12 30' '12 and'
 '12 hours' '12 minutes' '12th' '13' '13 minutes' '14' '15' '15 hours'
 '15 minutes' '150' '16' '17' '18' '18 years' '19' '1st' '20' '20 min'
 '20 mins' '20 minute' '20 minutes' '20 seconds' '20 years' '200' '2000'
 '21' '211' '22' '23' '24' '24 hours' '25']


## 9. Character-Level TF-IDF

Character-level TF-IDF captures patterns inside words.

This can be useful for personality prediction because it can capture:

- Word fragments
- Writing style patterns
- Common suffixes and prefixes
- Spelling variations
- Morphological patterns

We use character n-grams from 3 to 5 characters.

The vectorizer is again fitted only on the training dataset.

In [10]:
# Create a character-level TF-IDF vectorizer
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=3,
    max_features=15000,
    sublinear_tf=True
)

# Fit the character vocabulary only on training text
X_train_char = char_vectorizer.fit_transform(
    X_text_train
)

# Transform validation text
X_validation_char = char_vectorizer.transform(
    X_text_validation
)

# Transform test text
X_test_char = char_vectorizer.transform(
    X_text_test
)

# Display feature dimensions
print("Character-level TF-IDF")
print("-" * 50)

print("Train      :", X_train_char.shape)
print("Validation :", X_validation_char.shape)
print("Test       :", X_test_char.shape)

Character-level TF-IDF
--------------------------------------------------
Train      : (1578, 15000)
Validation : (395, 15000)
Test       : (494, 15000)


## 10. Inspect Character-Level Features

Let's inspect a sample of the character n-grams learned from the
training dataset.

In [11]:
# Get character-level feature names
char_features = char_vectorizer.get_feature_names_out()

# Display the total number of character features
print("Total Character TF-IDF Features:", len(char_features))

# Display a sample of learned character features
print("\nSample Character Features:")
print(char_features[:50])

Total Character TF-IDF Features: 15000

Sample Character Features:
[' " ' ' , ' ' - ' ' . ' ' . i' ' . i ' ' 10' ' 10 ' ' 12' ' 15' ' 2 '
 ' 20' ' 20 ' ' 20 m' ' 3 ' ' 30' ' 4 ' ' 5 ' ' a ' ' a b' ' a ba' ' a be'
 ' a bi' ' a bo' ' a bu' ' a c' ' a ca' ' a ch' ' a cl' ' a co' ' a d'
 ' a da' ' a de' ' a di' ' a do' ' a f' ' a fa' ' a fe' ' a fr' ' a g'
 ' a gi' ' a go' ' a gr' ' a gu' ' a h' ' a ha' ' a ho' ' a j' ' a jo'
 ' a l']


## 11. Create Statistical Text Features

TF-IDF captures textual patterns, but we can also extract simple
writing-style statistics.

The following features will be created:

- Character count
- Word count
- Sentence count
- Average word length
- Uppercase character ratio
- Digit ratio
- Punctuation ratio

These features provide additional information about the writing style.

In [12]:
def extract_text_statistics(text):
    """
    Extract basic statistical characteristics from a text document.

    Parameters
    ----------
    text : str
        Input document text.

    Returns
    -------
    list
        Numerical text statistics.
    """

    # Convert input to string
    text = str(text)

    # Calculate total number of characters
    character_count = len(text)

    # Extract words using a simple word pattern
    words = re.findall(r"\b\w+\b", text)

    # Calculate total number of words
    word_count = len(words)

    # Calculate average word length
    if word_count > 0:
        average_word_length = (
            sum(len(word) for word in words) / word_count
        )
    else:
        average_word_length = 0

    # Estimate sentence count using sentence-ending punctuation
    sentence_count = len(
        re.findall(r"[.!?]+", text)
    )

    # Count alphabetic characters
    alphabetic_characters = sum(
        character.isalpha()
        for character in text
    )

    # Count uppercase characters
    uppercase_characters = sum(
        character.isupper()
        for character in text
    )

    # Calculate uppercase ratio
    uppercase_ratio = (
        uppercase_characters / alphabetic_characters
        if alphabetic_characters > 0
        else 0
    )

    # Count digits
    digit_count = sum(
        character.isdigit()
        for character in text
    )

    # Calculate digit ratio
    digit_ratio = (
        digit_count / character_count
        if character_count > 0
        else 0
    )

    # Count punctuation characters
    punctuation_count = sum(
        not character.isalnum() and not character.isspace()
        for character in text
    )

    # Calculate punctuation ratio
    punctuation_ratio = (
        punctuation_count / character_count
        if character_count > 0
        else 0
    )

    return [
        character_count,
        word_count,
        sentence_count,
        average_word_length,
        uppercase_ratio,
        digit_ratio,
        punctuation_ratio
    ]

## 12. Test Statistical Feature Extraction

Before applying the function to the complete dataset,
test it on one example document.

In [13]:
# Select one training document
sample_text = X_text_train.iloc[0]

# Extract its statistical features
sample_statistics = extract_text_statistics(
    sample_text
)

# Display the result
print("Sample text statistics:")
print(sample_statistics)

Sample text statistics:
[3758, 773, 124, 3.6985769728331177, 0.0, 0.0, 0.03751995742416179]


## 13. Extract Statistical Features for All Splits

Apply the statistical feature extraction function to the
training, validation, and test datasets.

The same feature calculation is applied independently to each document.

In [14]:
# Extract statistical features from the training dataset
X_train_stats = np.array([
    extract_text_statistics(text)
    for text in X_text_train
])

# Extract statistical features from the validation dataset
X_validation_stats = np.array([
    extract_text_statistics(text)
    for text in X_text_validation
])

# Extract statistical features from the test dataset
X_test_stats = np.array([
    extract_text_statistics(text)
    for text in X_text_test
])

# Display feature dimensions
print("Statistical feature shapes")
print("-" * 50)

print("Train      :", X_train_stats.shape)
print("Validation :", X_validation_stats.shape)
print("Test       :", X_test_stats.shape)

Statistical feature shapes
--------------------------------------------------
Train      : (1578, 7)
Validation : (395, 7)
Test       : (494, 7)


## 14. Name the Statistical Features

Give each statistical feature a descriptive name so that the
feature-engineering process is easy to understand and document.

In [15]:
# Define names for the statistical text features
STAT_FEATURE_NAMES = [
    "character_count",
    "word_count",
    "sentence_count",
    "average_word_length",
    "uppercase_ratio",
    "digit_ratio",
    "punctuation_ratio"
]

# Display the feature names
print("Statistical Features:")
for feature in STAT_FEATURE_NAMES:
    print("-", feature)

Statistical Features:
- character_count
- word_count
- sentence_count
- average_word_length
- uppercase_ratio
- digit_ratio
- punctuation_ratio


## 15. Scale Statistical Features

The statistical features have different numerical ranges.

For example:

- Character count may be thousands.
- Word count may be hundreds.
- Ratios are usually between 0 and 1.

We therefore scale these features before combining them with TF-IDF.

`MaxAbsScaler` is used because it preserves sparse-matrix compatibility.

In [16]:
# Create the scaler for statistical features
stats_scaler = MaxAbsScaler()

# Fit the scaler ONLY on training statistical features
X_train_stats_scaled = stats_scaler.fit_transform(
    X_train_stats
)

# Transform validation statistical features
X_validation_stats_scaled = stats_scaler.transform(
    X_validation_stats
)

# Transform test statistical features
X_test_stats_scaled = stats_scaler.transform(
    X_test_stats
)

print("Statistical features scaled successfully.")

Statistical features scaled successfully.


## 16. Convert Statistical Features to Sparse Matrices

TF-IDF produces sparse matrices.

To combine TF-IDF and statistical features efficiently,
we convert the statistical features into sparse matrices.

In [17]:
# Convert scaled statistical features into sparse matrices
X_train_stats_sparse = csr_matrix(
    X_train_stats_scaled
)

X_validation_stats_sparse = csr_matrix(
    X_validation_stats_scaled
)

X_test_stats_sparse = csr_matrix(
    X_test_stats_scaled
)

print("Statistical features converted to sparse format.")

Statistical features converted to sparse format.


## 17. Combine NLP and Statistical Features

We now combine:

1. Word-level TF-IDF
2. Character-level TF-IDF
3. Statistical text features

The resulting feature matrix represents each document using
both linguistic and writing-style information.

In [18]:
# Combine word TF-IDF, character TF-IDF,
# and statistical features for the training set
X_train = hstack([
    X_train_word,
    X_train_char,
    X_train_stats_sparse
]).tocsr()

# Combine features for the validation set
X_validation = hstack([
    X_validation_word,
    X_validation_char,
    X_validation_stats_sparse
]).tocsr()

# Combine features for the test set
X_test = hstack([
    X_test_word,
    X_test_char,
    X_test_stats_sparse
]).tocsr()

# Display final feature dimensions
print("Combined Feature Matrix")
print("-" * 50)

print("Train      :", X_train.shape)
print("Validation :", X_validation.shape)
print("Test       :", X_test.shape)

Combined Feature Matrix
--------------------------------------------------
Train      : (1578, 35007)
Validation : (395, 35007)
Test       : (494, 35007)


## 18. Verify Feature Matrix Consistency

All three datasets must contain exactly the same number of features.

This is required because the same Machine Learning model will later
receive feature vectors with the same dimensions.

In [19]:
# Get the number of features in each dataset
train_features = X_train.shape[1]
validation_features = X_validation.shape[1]
test_features = X_test.shape[1]

# Display feature counts
print("Feature counts")
print("-" * 40)

print("Train      :", train_features)
print("Validation :", validation_features)
print("Test       :", test_features)

# Verify that all feature counts are identical
if not (
    train_features
    == validation_features
    == test_features
):
    raise ValueError(
        "Feature dimensions do not match across datasets."
    )

print("\nFeature dimensions are consistent.")

Feature counts
----------------------------------------
Train      : 35007
Validation : 35007
Test       : 35007

Feature dimensions are consistent.


## 19. Feature Composition

Let's calculate how many features come from each feature group.

This provides transparency into the final feature matrix.

In [20]:
# Count the number of features in each feature group
word_feature_count = X_train_word.shape[1]
char_feature_count = X_train_char.shape[1]
stat_feature_count = X_train_stats_sparse.shape[1]

# Display feature composition
feature_composition = pd.DataFrame({
    "Feature Type": [
        "Word TF-IDF",
        "Character TF-IDF",
        "Text Statistics"
    ],
    "Number of Features": [
        word_feature_count,
        char_feature_count,
        stat_feature_count
    ]
})

# Display the feature composition table
display(feature_composition)

# Display the total
print(
    "Total Features:",
    word_feature_count
    + char_feature_count
    + stat_feature_count
)

,Feature Type,Number of Features
0,Word TF-IDF,20000
1,Character TF-IDF,15000
2,Text Statistics,7


Total Features: 35007
